# Custom Method via `@register_method`

This notebook shows how to replace `method_id="cfm"` (which currently relies on
patches in `_cfm.py`) with a self-contained class defined right here.

`register_method` builds a new class at decoration time:

```
RegisteredMethod = type(name, (YourClass, TorchGenerativeFlow), {
    "_step_fn":  wraps YourClass.step_fn,
    "_predict":  wraps YourClass.predict,   # see note in class docstring
    "__init__":  calls both inits,
})
METHODS_REGISTRY[name] = RegisteredMethod
```

**MRO:** `RegisteredMethod -> YourClass -> TorchGenerativeFlow -> ...`

Because `YourClass` is first, any method you define **shadows** the base class
method of the same name.  This matters most for `predict` — see below.

In [ ]:
import torch

from sc_flow import SCFlow
from sc_flow.methods import register_method
from sc_flow.backends.torch.methods import METHODS_REGISTRY
from sc_flow.backends.torch.methods._base import TorchGenerativeFlow
from sc_flow.backends.torch.methods._utils import StepData
from sc_flow.backends.torch.nn._vf import MLPVelocity
from sc_flow.backends.torch.solvers import ODESolver
from sc_flow.backends.torch._types import PredictionData

In [ ]:
# Guard: re-running this cell crashes without this check because
# METHODS_REGISTRY["contextflow"] already exists after the first run.
if "contextflow" not in METHODS_REGISTRY:

    @register_method("contextflow", backend="torch", category="flow")
    class ContextFlow:
        """
        ContextFlow: CFM with multi-transition training.

        CONTRACT for category="flow"
        ─────────────────────────────────────────────────────────────────────
        step_fn(self, step_data: StepData)
            Required by register_method — it checks hasattr(user_cls, "step_fn")
            at decoration time and wires it to _step_fn.
            Here it is a stub because train_step always routes to
            _train_step_multi (we always use MultiTransitionSampler).

        predict(self, matched_distr, ...)
            Required. Takes MatchedDistributions — NOT StepData.
            Because ContextFlow is first in the MRO, this method SHADOWS
            TorchGenerativeFlow.predict entirely.  SCFlow calls
            method.predict(node) directly, so you receive MatchedDistributions
            and must call self._extract_step_data() yourself.

            (The _predict wrapper that register_method adds to the class dict
            is never actually called — TorchGenerativeFlow.predict, the only
            thing that invokes _predict, is now shadowed by this method.)

        Anything else you define (train_step, _train_step_multi, __init__)
        is inherited directly via the MRO and shadows the base class version.
        """

        module_cls = MLPVelocity  # velocity field architecture
        default_solver_cls = ODESolver  # ODE solver for inference

        # ── __init__ ────────────────────────────────────────────────────────
        # Runs AFTER TorchGenerativeFlow.__init__ (which sets _noise_sampler,
        # _time_sampler, _probability_path, _match_fn to whatever was passed
        # to SCFlow, defaulting to None).  Set fallback defaults here.
        def __init__(self, *args, **kwargs):
            if self._noise_sampler is None:
                self._noise_sampler = torch.randn_like
            if self._time_sampler is None:
                self._time_sampler = torch.rand

        # ── step_fn (stub) ──────────────────────────────────────────────────
        # Required by register_method(category="flow") — never actually called
        # because train_step always dispatches to _train_step_multi.
        def step_fn(self, step_data: StepData, **kwargs):
            """Stub: single-transition path is unused; train_step always routes to _train_step_multi."""
            raise NotImplementedError("single-transition path not used")

        # ── predict ─────────────────────────────────────────────────────────
        # Full public predict — takes MatchedDistributions, not StepData.
        # Must mirror the signature that callers (SCFlow, model.predict())
        # expect, including no_grad, solver_kwargs, return_trajectory, etc.
        def predict(
            self,
            matched_distr,
            *args,
            no_grad=True,
            solver_cls=None,
            solver_kwargs=None,
            return_trajectory=False,
            num_steps=100,
            latent=None,
            t_start=0.0,
            t_end=1.0,
            **kwargs,
        ):
            """Run ODE inference from MatchedDistributions using the registered solver."""
            # extract tensors — this is what TorchGenerativeFlow.predict
            # normally does before calling _predict
            step_data = self._extract_step_data(matched_distr)

            def _run():
                nonlocal latent
                if latent is None:
                    src, tgt = step_data.source_state, step_data.target_state
                    latent = src if (src is not None and not self._generate_from_noise) else self._noise_sampler(tgt)

                cond_dict = {
                    **self._get_tensor_dict_from_data(step_data.target_condition_data),
                    **self._get_tensor_dict_from_data(step_data.target_group_data),
                }

                _skw = {} if solver_kwargs is None else dict(solver_kwargs)
                _skw.setdefault("method", "euler")
                method = _skw.pop("method")

                _scls = self._default_solver_cls if solver_cls is None else solver_cls
                time_grid = torch.linspace(
                    t_start,
                    t_end,
                    steps=num_steps,
                    device=latent.device,
                    dtype=latent.dtype,
                )
                solver = _scls(
                    self._module,
                    method=method,
                    vf_kwargs={"condition_dict": cond_dict, "source": step_data.source_state},
                    device_id=self._device_id,
                )
                preds = solver.solve(
                    latent,
                    time_grid,
                    solver_kwargs=_skw,
                    return_trajectory=return_trajectory,
                )
                if return_trajectory:
                    return PredictionData(preds[-1], traj=preds)
                return PredictionData(preds, traj=None)

            if no_grad:
                with torch.no_grad():
                    return _run()
            return _run()

        # ── train_step ──────────────────────────────────────────────────────
        # Shadows TorchGenerativeFlow.train_step.
        # The trainer calls this with transitions=True + timepoints=[...]
        # when using MultiTransitionSampler.
        def train_step(self, matched_distr_or_nodes, *args, transitions=False, **kwargs):
            """Dispatch to multi-transition or single-transition training step."""
            if transitions:
                return self._train_step_multi(matched_distr_or_nodes, *args, **kwargs)
            return TorchGenerativeFlow.train_step(self, matched_distr_or_nodes, *args, **kwargs)

        # ── _train_step_multi ────────────────────────────────────────────────
        # The real training logic. Collects all transitions into one
        # concatenated forward pass so the model sees the full time axis
        # in a single gradient step.
        def _train_step_multi(self, nodes, *args, timepoints, **kwargs):
            all_t, all_xt, all_ut, all_cond, all_latent = [], [], [], [], []

            for i, node in enumerate(nodes):
                step_data = self._extract_step_data(node)
                step_data = self._extract_matched_observations(step_data)  # OT per transition

                target = step_data.target_state
                source = step_data.source_state
                condition_data = self._get_tensor_dict_from_data(step_data.target_condition_data)
                group_data = self._get_tensor_dict_from_data(step_data.target_group_data)

                latent = (
                    source if (source is not None and not self._generate_from_noise) else self._noise_sampler(target)
                )
                B = latent.shape[0]

                # local t in [0, 1] is rescaled to the global interval [t_i, t_{i+1}]
                t_start_i, t_end_i = timepoints[i], timepoints[i + 1]
                delta_t = t_end_i - t_start_i
                t_local = self._time_sampler((B,), device=latent.device, dtype=latent.dtype)
                t_global = t_local * delta_t + t_start_i

                xt = self._probability_path.compute_xt(t_local, latent, target)
                ut = self._probability_path.compute_ut(t_local, xt, latent, target) / delta_t

                all_t.append(t_global)
                all_xt.append(xt)
                all_ut.append(ut)
                all_cond.append({**condition_data, **group_data})
                all_latent.append(latent)

            # single forward pass over all transitions concatenated
            t_all = torch.cat(all_t)
            xt_all = torch.cat(all_xt)
            ut_all = torch.cat(all_ut)
            latent_all = torch.cat(all_latent)
            cond_all = {k: torch.cat([c[k] for c in all_cond]) for k in all_cond[0]}

            vt_all = self._module(t_all, xt_all, condition_dict=cond_all, source=latent_all)
            loss = torch.nn.functional.mse_loss(vt_all, ut_all)
            return loss, {"loss": loss.item()}

In [ ]:
# Drop-in replacement for method_id="cfm".
# Pass probability_path here — probability_path_cls is not usable because
# LinearGaussianProbabilityPath requires arguments (sigma, prng).
model = SCFlow(
    method_id="contextflow",
    backend="torch",
    match_fn=ctf_c_partial,  # noqa: F821
    probability_path=probability_path,  # noqa: F821
    vf_decoder_mlp_kwargs={"hidden_dims": hidden_dims, "activation_cls": activation},  # noqa: F821
)